<a href="https://colab.research.google.com/github/SurajReddy469/2520030355__ML/blob/main/Project/Personalized_Recommendation_Final_EDA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Personalized ML Based Recommendation System — Final EDA

This notebook uses the official **MovieLens ml-latest-small** dataset.

Required datasets:
- `movies.csv`
- `ratings.csv`

The notebook downloads and extracts the dataset automatically in Google Colab.


In [ ]:
# ============================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
import urllib.request
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("Libraries imported successfully!")


In [ ]:
# ============================================
# 2. DOWNLOAD AND EXTRACT MOVIELENS DATASET
# ============================================

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "/content/ml-latest-small.zip"
extract_path = "/content"

if not os.path.exists("/content/ml-latest-small"):
    urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

print("Dataset ready!")
print(os.listdir("/content/ml-latest-small"))


In [ ]:
# ============================================
# 3. LOAD DATASETS
# ============================================

movies = pd.read_csv("/content/ml-latest-small/movies.csv")
ratings = pd.read_csv("/content/ml-latest-small/ratings.csv")

print("Movies Dataset:")
display(movies.head())

print("\nRatings Dataset:")
display(ratings.head())


In [ ]:
# ============================================
# 4. DATASET OVERVIEW
# ============================================

print("Movies Shape :", movies.shape)
print("Ratings Shape:", ratings.shape)

print("Unique Users:", ratings["userId"].nunique())
print("Unique Movies Rated:", ratings["movieId"].nunique())
print("Total Movies:", movies["movieId"].nunique())


In [ ]:
# ============================================
# 5. DATA TYPES
# ============================================

print("Movies Data Types:")
display(movies.dtypes)

print("\nRatings Data Types:")
display(ratings.dtypes)


In [ ]:
# ============================================
# 6. MISSING VALUES AND DUPLICATES
# ============================================

print("Missing values in Movies:")
display(movies.isnull().sum())

print("\nMissing values in Ratings:")
display(ratings.isnull().sum())

print("\nDuplicate rows in Movies:", movies.duplicated().sum())
print("Duplicate rows in Ratings:", ratings.duplicated().sum())


In [ ]:
# ============================================
# 7. RATING STATISTICS
# ============================================

display(ratings["rating"].describe())

print("\nRating Value Counts:")
display(ratings["rating"].value_counts().sort_index())


In [ ]:
# ============================================
# 8. RATING DISTRIBUTION
# ============================================

plt.figure(figsize=(10, 6))

sns.countplot(
    data=ratings,
    x="rating",
    order=sorted(ratings["rating"].unique())
)

plt.title("Movie Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 9. GENRE ANALYSIS
# ============================================

genre_counts = (
    movies["genres"]
    .fillna("")
    .str.split("|")
    .explode()
    .replace("", np.nan)
    .dropna()
    .value_counts()
)

display(genre_counts)


In [ ]:
# ============================================
# 10. GENRE DISTRIBUTION
# ============================================

plt.figure(figsize=(12, 6))
genre_counts.plot(kind="bar")

plt.title("Movie Genre Distribution")
plt.xlabel("Genre")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 11. USER ACTIVITY ANALYSIS
# ============================================

user_stats = ratings.groupby("userId").agg(
    rating_count=("rating", "count"),
    average_rating=("rating", "mean")
)

print("User Statistics:")
display(user_stats.describe())

print("\nTop 10 Users by Number of Ratings:")
display(user_stats.sort_values("rating_count", ascending=False).head(10))


In [ ]:
# ============================================
# 12. RATINGS PER USER
# ============================================

plt.figure(figsize=(10, 6))

sns.histplot(
    user_stats["rating_count"],
    bins=30,
    kde=True
)

plt.title("Number of Ratings per User")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Users")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 13. MOVIE POPULARITY
# ============================================

movie_stats = ratings.groupby("movieId").agg(
    rating_count=("rating", "count"),
    average_rating=("rating", "mean")
).reset_index()

movie_stats = movie_stats.merge(
    movies,
    on="movieId",
    how="left"
)

print("Most Rated Movies:")
display(
    movie_stats
    .sort_values("rating_count", ascending=False)
    [["movieId", "title", "rating_count", "average_rating"]]
    .head(10)
)


In [ ]:
# ============================================
# 14. HIGHEST-RATED POPULAR MOVIES
# ============================================

popular_movies = movie_stats[
    movie_stats["rating_count"] >= 50
].sort_values(
    "average_rating",
    ascending=False
)

display(
    popular_movies[
        ["movieId", "title", "rating_count", "average_rating"]
    ].head(10)
)


In [ ]:
# ============================================
# 15. MERGE MOVIES AND RATINGS
# ============================================

movie_ratings = ratings.merge(
    movies,
    on="movieId",
    how="left"
)

print("Merged Dataset Shape:", movie_ratings.shape)
display(movie_ratings.head())


In [ ]:
# ============================================
# 16. TF-IDF CONTENT FEATURES
# ============================================

movies_tfidf = movies.copy()

movies_tfidf["genres"] = (
    movies_tfidf["genres"]
    .fillna("")
    .str.replace("|", " ", regex=False)
)

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies_tfidf["genres"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)


In [ ]:
# ============================================
# 17. COSINE SIMILARITY
# ============================================

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)


In [ ]:
# ============================================
# 18. CONTENT-BASED RECOMMENDATION FUNCTION
# ============================================

indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

def recommend_movies(title, num_recommendations=10):

    if title not in indices:
        print("Movie not found.")
        return

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[
        1:num_recommendations + 1
    ]

    movie_indices = [i[0] for i in similarity_scores]

    result = movies.iloc[movie_indices][
        ["movieId", "title", "genres"]
    ].copy()

    result["similarity_score"] = [
        round(i[1], 4)
        for i in similarity_scores
    ]

    return result


In [ ]:
# ============================================
# 19. TEST RECOMMENDATION
# ============================================

recommend_movies("Toy Story (1995)", 10)


In [ ]:
# ============================================
# 20. CREATE USER AND MOVIE FEATURES
# ============================================

user_features = ratings.groupby("userId").agg(
    user_mean_rating=("rating", "mean"),
    user_rating_count=("rating", "count")
).reset_index()

movie_features = ratings.groupby("movieId").agg(
    movie_mean_rating=("rating", "mean"),
    movie_rating_count=("rating", "count")
).reset_index()

display(user_features.head())
display(movie_features.head())


In [ ]:
# ============================================
# 21. CREATE ML DATASET
# ============================================

model_data = ratings.merge(
    user_features,
    on="userId",
    how="left"
).merge(
    movie_features,
    on="movieId",
    how="left"
)

features = [
    "user_mean_rating",
    "user_rating_count",
    "movie_mean_rating",
    "movie_rating_count"
]

X = model_data[features]
y = model_data["rating"]

print("ML Dataset Shape:", model_data.shape)
display(model_data.head())


In [ ]:
# ============================================
# 22. TRAIN RANDOM FOREST
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Records:", len(X_train))
print("Testing Records :", len(X_test))

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest training completed!")


In [ ]:
# ============================================
# 23. MODEL EVALUATION
# ============================================

predictions = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MODEL PERFORMANCE")
print("=" * 40)
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")


In [ ]:
# ============================================
# 24. FINAL EDA SUMMARY
# ============================================

print("=" * 60)
print("PERSONALIZED ML RECOMMENDATION SYSTEM")
print("FINAL EDA SUMMARY")
print("=" * 60)

print(f"Total Movies        : {len(movies):,}")
print(f"Total Ratings       : {len(ratings):,}")
print(f"Unique Users        : {ratings['userId'].nunique():,}")
print(f"Rated Movies        : {ratings['movieId'].nunique():,}")
print(f"Average Rating      : {ratings['rating'].mean():.2f}")
print(f"Minimum Rating      : {ratings['rating'].min():.1f}")
print(f"Maximum Rating      : {ratings['rating'].max():.1f}")
print(f"Number of Genres    : {len(genre_counts):,}")
print(f"TF-IDF Features     : {tfidf_matrix.shape[1]:,}")

print("\nEDA completed successfully!")
